# Advanced Exploratory Data Analysis (EDA) of PTB-XL Dataset

This notebook provides an advanced visualization of the PTB-XL ECG dataset. It covers the exploration of clinical demographics, evaluates the class imbalances of the diagnostic superclasses, and offers an advanced visual comparison of the 12-lead ECG waveforms.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import wfdb
import ast
import os

# Set high-end visual aesthetics
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({'font.size': 12})

## 1. Load Metadata
First, we load the curated metadata `results/ptbxl_metadata_with_superclasses.csv` that contains labels, and extract necessary representations.

In [ ]:
DATA_DIR = '../data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/'
META_PATH = '../results/ptbxl_metadata_with_superclasses.csv'

df = pd.read_csv(META_PATH)

# Parse the scp_codes column from string to an actual python dictionary
df['scp_codes'] = df['scp_codes'].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else {})

print(f"Total clinical records available: {len(df)}")
df.head(3)

## 2. Demographics Distribution
Analyzing the basic demographics such as the age and sex of the patients in the dataset.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age Distribution
sns.histplot(data=df, x='age', bins=30, kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Age Distribution of Patients', fontweight='bold')
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Count')

# Sex Distribution
# In the PTB-XL dataset representation, generally 0 denotes Male, and 1 denotes Female
sex_mapping = {0: 'Male', 1: 'Female'}
df['sex_label'] = df['sex'].map(sex_mapping).fillna('Unknown')
sns.countplot(data=df, x='sex_label', ax=axes[1], palette='pastel')
axes[1].set_title('Sex Distribution', fontweight='bold')
axes[1].set_xlabel('Sex')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 3. Diagnostic Superclass Distibution & Class Imbalance
We investigate the counts of the main diagnostic superclasses: 
1. **NORM**: Normal ECG
2. **MI**: Myocardial Infarction
3. **STTC**: ST/T Change
4. **CD**: Conduction Disturbance
5. **HYP**: Hypertrophy

In [ ]:
superclasses = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
class_counts = df[superclasses].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
ax = sns.barplot(x=class_counts.index, y=class_counts.values, palette='viridis')
plt.title('Distribution of Diagnostic Superclasses (Class Imbalance)', fontweight='bold')
plt.xlabel('Diagnostic Superclass')
plt.ylabel('Number of Occurrences / Patients')

# Add percentage labels for better clarity
total = len(df)
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.1f}%'
    ax.annotate(percentage, (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11, color='blue', xytext=(0, 5),
                textcoords='offset points', fontweight='bold')

plt.show()

## 4. Advanced ECG Signal Visualization
Plotting a comprehensive visual summary of the 12 standard leads across a specific sample allows domain experts to recognize anomalies (such as an inverted T-wave or ST-segment elevation).

In [ ]:
def plot_12_lead_ecg(ecg_id, df, data_dir, freq='100'):
    """Plots a standard 12-lead ECG layout. 
    Grid is superimposed as red to mimic real ECG paper."""
    row = df[df['ecg_id'] == ecg_id].iloc[0]
    filename = row['filename_lr'] if freq == '100' else row['filename_hr']
    path = os.path.join(data_dir, filename)
    
    try:
        signal, meta = wfdb.rdsamp(path)
    except FileNotFoundError:
        print(f"File {path} not found. Please ensure dataset is correctly extracted.")
        return
        
    leads = meta['sig_name']
    
    # 4 columns x 3 rows is standard presentation for 12 leads
    fig, axes = plt.subplots(3, 4, figsize=(18, 10), sharex=True, sharey=True)
    fig.suptitle(f"12-Lead ECG - Record {ecg_id} (Superclasses: {row['diagnostic_superclass']})", 
                 fontsize=16, fontweight='bold', y=0.98)
    
    layout = [
        ['I', 'AVR', 'V1', 'V4'],
        ['II', 'AVL', 'V2', 'V5'],
        ['III', 'AVF', 'V3', 'V6']
    ]
    
    time = np.arange(signal.shape[0]) / meta['fs']
    
    for i in range(3):
        for j in range(4):
            lead_name = layout[i][j]
            if lead_name in leads:
                lead_idx = leads.index(lead_name)
                # Add grid to imitate standardized ECG paper
                axes[i, j].grid(True, which='both', color='red', alpha=0.3)
                axes[i, j].plot(time, signal[:, lead_idx], color='black', linewidth=1)
                # Set clear titles, e.g V1, V2 etc.
                axes[i, j].set_title(lead_name, loc='left', color='darkblue', fontweight='bold', fontsize=12)
                
                if i == 2:
                    axes[i, j].set_xlabel("Time (s)")
                if j == 0:
                    axes[i, j].set_ylabel("Amplitude (mV)")
                    
    plt.tight_layout()
    plt.subplots_adjust(top=0.92)
    plt.show()

# Let's visualize a clean 'Healthy' sample
norm_sample = df[df['NORM'] == 1]['ecg_id'].sample(1, random_state=42).iloc[0]
plot_12_lead_ecg(norm_sample, df, DATA_DIR, freq='100')

# Let's visualize an 'Infarction' sample to look for noticeable pattern deviations
mi_sample = df[df['MI'] == 1]['ecg_id'].sample(1, random_state=42).iloc[0]
plot_12_lead_ecg(mi_sample, df, DATA_DIR, freq='100')